<a href="https://colab.research.google.com/github/ravindravala/AI/blob/main/student_dropout_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:

import kagglehub
from kagglehub import KaggleDatasetAdapter
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset


file_path = "student_dropout_dataset_v3.csv"

df = kagglehub.dataset_load(
  KaggleDatasetAdapter.PANDAS,
  "meharshanali/student-dropout-prediction-dataset",
  file_path
)


Using Colab cache for faster access to the 'student-dropout-prediction-dataset' dataset.


In [27]:
df

,Student_ID,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education,Dropout
0,1,22.1,Male,25000.0,Yes,3.36,86.1,2,20.4,Yes,No,5.5,0.96,0.90,0.90,Year 1,Arts,High School,0
1,2,20.7,Male,25000.0,Yes,4.30,68.0,2,44.0,No,No,6.8,1.28,1.20,1.19,Year 3,Engineering,Bachelor,1
2,3,22.4,Male,40183.0,Yes,4.40,70.9,0,48.9,Yes,No,5.5,1.68,1.32,1.32,Year 1,Arts,Master,0
3,4,24.4,Male,NaN,Yes,NaN,82.2,2,38.6,No,No,NaN,1.78,1.77,1.77,Year 1,CS,High School,1
4,5,20.5,Female,25319.0,Yes,4.19,75.7,1,23.0,No,No,7.0,1.48,0.91,0.87,Year 4,Business,Bachelor,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,23.9,Female,42286.0,No,4.62,92.0,0,10.0,Yes,Yes,5.5,1.60,0.99,0.97,Year 2,Arts,Bachelor,0
9996,9997,17.0,Female,61103.0,Yes,2.87,75.2,3,32.4,No,Yes,6.7,3.09,3.09,3.09,Year 1,Business,Master,1
9997,9998,19.4,Male,25000.0,Yes,4.73,74.9,4,25.4,No,No,3.5,3.45,3.37,3.43,Year 4,Business,Bachelor,0
9998,9999,22.1,Female,40302.0,Yes,5.85,74.2,1,5.0,No,Yes,6.2,3.35,3.34,3.34,Year 1,CS,High School,0


In [28]:
x = df.drop(columns=['Dropout','Student_ID'],axis=1)
y = df['Dropout']

In [29]:
x['Gender'] = x['Gender'].map({'Male': 0, 'Female': 1})
x['Internet_Access'] = x['Internet_Access'].map({'No': 0, 'Yes': 1})
x['Part_Time_Job'] = x['Part_Time_Job'].map({'No': 0, 'Yes': 1})
x['Scholarship'] = x['Scholarship'].map({'No': 0, 'Yes': 1})


In [30]:
# numeric columns
x = x.fillna(x.mean(numeric_only=True))

# categorical columns
x = x.fillna("Unknown")

In [31]:
x

,Age,Gender,Family_Income,Internet_Access,Study_Hours_per_Day,Attendance_Rate,Assignment_Delay_Days,Travel_Time_Minutes,Part_Time_Job,Scholarship,Stress_Index,GPA,Semester_GPA,CGPA,Semester,Department,Parental_Education
0,22.1,0,25000.000000,1,3.360000,86.1,2,20.4,1,0,5.500000,0.96,0.90,0.90,Year 1,Arts,High School
1,20.7,0,25000.000000,1,4.300000,68.0,2,44.0,0,0,6.800000,1.28,1.20,1.19,Year 3,Engineering,Bachelor
2,22.4,0,40183.000000,1,4.400000,70.9,0,48.9,1,0,5.500000,1.68,1.32,1.32,Year 1,Arts,Master
3,24.4,0,38377.247474,1,4.014592,82.2,2,38.6,0,0,5.507147,1.78,1.77,1.77,Year 1,CS,High School
4,20.5,1,25319.000000,1,4.190000,75.7,1,23.0,0,0,7.000000,1.48,0.91,0.87,Year 4,Business,Bachelor
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,23.9,1,42286.000000,0,4.620000,92.0,0,10.0,1,1,5.500000,1.60,0.99,0.97,Year 2,Arts,Bachelor
9996,17.0,1,61103.000000,1,2.870000,75.2,3,32.4,0,1,6.700000,3.09,3.09,3.09,Year 1,Business,Master
9997,19.4,0,25000.000000,1,4.730000,74.9,4,25.4,0,0,3.500000,3.45,3.37,3.43,Year 4,Business,Bachelor
9998,22.1,1,40302.000000,1,5.850000,74.2,1,5.0,0,1,6.200000,3.35,3.34,3.34,Year 1,CS,High School


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

In [33]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), ['Semester','Department','Parental_Education']),
        ('num',StandardScaler(),['Family_Income'])
    ])

In [34]:
x = preprocessor.fit_transform(x)

In [35]:
x = x.toarray()

In [36]:
x = torch.tensor(x, dtype=torch.float32)
y = torch.tensor(y.values, dtype=torch.float32)

In [37]:
class StudentData(Dataset):
  def __init__(self, x,y):
    self.x = x
    self.y = y

  def __len__(self):
    return len(self.x)

  def __getitem__(self, idx):
    return self.x[idx], self.y[idx]

In [38]:
dataset = StudentData(x,y)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [63]:

model = nn.Sequential(
    nn.Linear(x.shape[1], 128),
    nn.ReLU(),
    nn.Linear(128, 64),
    nn.ReLU(),
    nn.Linear(64, 1)
)

optimizer = optim.Adam(model.parameters(), lr=0.05)
pos_weight = (len(y) - y.sum()) / y.sum()

loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

In [64]:
for epoch in range(100):
    total_loss = 0

    for X_batch, y_batch in loader:

        y_batch = y_batch.view(-1, 1).float()

        outputs = model(X_batch)
        loss = loss_fn(outputs, y_batch)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader)}")

Epoch 1, Loss: 1.0688239697831126
Epoch 11, Loss: 1.0624096743976728
Epoch 21, Loss: 1.0613839972895174
Epoch 31, Loss: 1.061569429815006
Epoch 41, Loss: 1.0624208248461398
Epoch 51, Loss: 1.0622066552646625
Epoch 61, Loss: 1.0636684839337016
Epoch 71, Loss: 1.0618865615643633
Epoch 81, Loss: 1.0613826303817213
Epoch 91, Loss: 1.0627494679091456


In [66]:
model.eval()   # switch to eval mode

correct = 0
total = 0

with torch.no_grad():   # no gradients needed

    for X_batch, y_batch in loader:

        y_batch = y_batch.view(-1, 1).float()

        outputs = model(X_batch)

        preds = torch.sigmoid(outputs)       # convert logits → probabilities
        preds = (preds > 0.5).float()        # threshold → 0 or 1

        correct += (preds == y_batch).sum().item()
        total += y_batch.numel()

accuracy = correct / total
print("Accuracy:", accuracy)

Accuracy: 0.7646


In [61]:
print(x[:5])
print(x.shape)
print(model)

tensor([[ 1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00, -6.6966e-01],
        [ 0.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00, -6.6966e-01],
        [ 1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,
          0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  9.0395e-02],
        [ 1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          0.0000e+00,  1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00,  3.6423e-16],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  1.0000e+00,  0.0000e+00,
          1.0000e+00,  0.0000e+00,

In [62]:
print(y.mean())

tensor(0.2354)
